# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset consists of survey data collected from residents of Kilifi County, focusing on mental health indicators, demographics, and clinical assessment scores (GAD-7, PHQ-9, and PSQ).

## Dataset Source
The dataset metadata is defined by a Croissant schema and is accessible via the following URL:
<br>
<code>https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json</code>

In [ ]:
# Ensure `mlcroissant` library is installed!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display high-level summary
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')} | Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets, their fields, and associated `@id` values.

Entities in Croissant datasets are referenced by their unique `@id`. We'll list the record sets, their IDs, and their corresponding field/column IDs.

In [ ]:
from mlcroissant._src.structure.graph import find_entities

# The metadata.record_set property gives a list of RecordSet objects
record_sets = getattr(metadata, 'record_set', [])
if not record_sets:
    # Some schemas place record sets under 'recordsets' (plural) or need graph traversal
    # Use mlcroissant low-level API to traverse the graph
    record_sets = find_entities(dataset.graph, "cr:RecordSet")

print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for idx, rs in enumerate(record_sets):
    rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
    rs_name = getattr(rs, 'name', None) or getattr(rs, 'label', None)
    print(f"{idx+1}. RecordSet @id: {rs_id}")
    print(f"   Name: {rs_name}")
    # Gather field/column information
    fields = getattr(rs, 'field', None)
    if fields is None:
        # Some schemas may use 'column' instead
        fields = getattr(rs, 'column', None)
    if fields:
        if not isinstance(fields, list):
            fields = [fields]
        print("   Fields/Columns @id:")
        for f in fields:
            try:
                fname = getattr(f, 'name', None) or getattr(f, 'label', None)
            except Exception:
                fname = None
            fid = getattr(f, '@id', None) or getattr(f, 'id', None) or f
            print(f"     - {fid} ({fname})")
    else:
        print("   No field/column info on this RecordSet.")
    print()
    if rs_id is not None:
        record_set_ids.append(rs_id)

print("RecordSet @id list for use in extraction:")
print(record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# We'll extract all record sets into dataframes for exploration
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# If no record sets found above, manually set the @id for the main survey RecordSet
if not record_set_ids:
    # You may want to look in the JSON or Croissant source
    # For demonstration, we use the known @id from the record set in the latest FAIR2 mental health exported schema
    record_set_ids = [
        "http://senscience.ai/frontiers/7160411/records"
    ]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if not records:
            print(f"  No records found for {rs_id}")
            continue
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Loaded {len(dataframes[rs_id])} records. Columns:")
        print(f"    {dataframes[rs_id].columns.tolist()}\n")
    except Exception as e:
        print(f"  Error loading records for {rs_id}: {e}")

# Pick the main DataFrame for EDA
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Perform common data processing: filter records, normalize numeric fields, and group by categorical variables. All columns are referenced by their Croissant `@id` (see previous overview step).

In [ ]:
# Identify column ids for numeric fields: e.g., GAD-7 total, PHQ-9 total, PSQ total
# You may check columns in main_df.columns
print("All columns in main record set:")
print(main_df.columns.tolist())

# Let's pick GAD-7 total score for analysis. Assume the field '@id' is 'gad7_total_score'
# If columns are named as per field '@id'. Adjust if the column names have a URI prefix etc.
numeric_field = None
for c in main_df.columns:
    if "gad" in c.lower() and ("score" in c.lower() or "total" in c.lower()):
        numeric_field = c
        break

if not numeric_field:
    # If not found, use PHQ-9
    for c in main_df.columns:
        if "phq" in c.lower() and ("score" in c.lower() or "total" in c.lower()):
            numeric_field = c
            break

print(f"Using numeric field: {numeric_field}\n")

# Set a threshold for filtering
threshold = 10

filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a demographic field, e.g., gender (find suitable field)
group_field = None
for c in main_df.columns:
    if 'gender' in c.lower():
        group_field = c
        break
if not group_field:
    for c in main_df.columns:
        if 'sex' in c.lower():
            group_field = c
            break

if group_field:
    grouped_df = (
        filtered_df.groupby(group_field)[numeric_field]
        .mean()
        .reset_index()
        .rename(columns={numeric_field: f"mean_{numeric_field}"})
    )
    print(f"\nMean {numeric_field} by {group_field}:")
    print(grouped_df)
else:
    print("No gender/sex field found to group by.")

## 5. Visualization

Visualize the distribution of the chosen numeric field, and show average values by group (e.g., by gender). If using this notebook in a new environment, you may have to install matplotlib/plotly.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Histogram of the numeric field (filtered and unfiltered)
plt.figure(figsize=(7, 4))
main_df[numeric_field].hist(bins=15, alpha=0.5, label='All')
filtered_df[numeric_field].hist(bins=15, alpha=0.7, color='orange', label=f'>{threshold}')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.legend()
plt.show()

# Barplot of mean score by group
if group_field:
    plt.figure(figsize=(5,3))
    plt.bar(grouped_df[group_field], grouped_df[f"mean_{numeric_field}"])
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion

- We loaded a complex Croissant metadata-driven dataset with `mlcroissant` using only the Croissant schema URL, following schema `@id` links for all entities.
- We explored available record sets and fields using Croissant metadata, extracted data to pandas DataFrames, and performed basic filtering and normalization using dynamically resolved Croissant column IDs.
- Initial EDA suggests the distribution of GAD-7 (or PHQ-9) total scores across participants, with further breakdowns (e.g., by gender) possible using the standardized and FAIR data structure.

You can extend this notebook for further analysis, modeling, or visualization as needed, referencing dataset fields by their `@id` for robust, schema-driven data processing.